In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

# Check dataset path
train_path = "New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train"
valid_path = "New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid"

# Check how many disease classes we have
classes = os.listdir(train_path)
print(f"Total disease classes: {len(classes)}")
print("\nDisease types:")
for c in classes[:10]:
    print(f"  - {c}")

Total disease classes: 3

Disease types:
  - Apple___Apple_scab
  - Apple___Black_rot
  - Apple___Cedar_apple_rust


In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Image settings
IMG_SIZE = 224
BATCH_SIZE = 32

# Load training images
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

valid_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

valid_data = valid_datagen.flow_from_directory(
    valid_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print("\n✅ Data loaded!")
print(f"Classes: {train_data.class_indices}")
print(f"Training images: {train_data.samples}")
print(f"Validation images: {valid_data.samples}")

Found 5517 images belonging to 3 classes.


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid'

In [3]:
import os

# Find the correct valid path
base = "New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)"

print("Folders inside dataset:")
for folder in os.listdir(base):
    print(f"  - {folder}")

Folders inside dataset:
  - train


In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 224
BATCH_SIZE = 32

# Use train data with 20% split for validation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 20% for validation
)

train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

valid_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

print("\n✅ Data loaded!")
print(f"Classes: {train_data.class_indices}")
print(f"Training images: {train_data.samples}")
print(f"Validation images: {valid_data.samples}")

Found 4415 images belonging to 3 classes.
Found 1102 images belonging to 3 classes.

✅ Data loaded!
Classes: {'Apple___Apple_scab': 0, 'Apple___Black_rot': 1, 'Apple___Cedar_apple_rust': 2}
Training images: 4415
Validation images: 1102


In [5]:

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout

# Use MobileNetV2 as base (pretrained on millions of images!)
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

# Build model
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dense(3, activation='softmax')  # 3 disease classes
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model built!")
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 5s 1us/step
✅ Model built!


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Stop training if no improvement
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

# Save best model
checkpoint = ModelCheckpoint(
    'best_model.keras',
    monitor='val_accuracy',
    save_best_only=True
)

print("🚀 Training started...")
print("This will take 10-15 minutes...")

history = model.fit(
    train_data,
    epochs=10,
    validation_data=valid_data,
    callbacks=[early_stop, checkpoint]
)

print("\n✅ Training complete!")

🚀 Training started...
This will take 10-15 minutes...
Epoch 1/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 348s 2s/step - accuracy: 0.9033 - loss: 0.2454 - val_accuracy: 0.9746 - val_loss: 0.0755
Epoch 2/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 315s 2s/step - accuracy: 0.9540 - loss: 0.1193 - val_accuracy: 0.9728 - val_loss: 0.0852
Epoch 3/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 272s 2s/step - accuracy: 0.9529 - loss: 0.1210 - val_accuracy: 0.9610 - val_loss: 0.1095
Epoch 4/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 286s 2s/step - accuracy: 0.9601 - loss: 0.0992 - val_accuracy: 0.9764 - val_loss: 0.0547
Epoch 5/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 242s 2s/step - accuracy: 0.9669 - loss: 0.0895 - val_accuracy: 0.9710 - val_loss: 0.0818
Epoch 6/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 251s 2s/step - accuracy: 0.9696 - loss: 0.0833 - val_accuracy: 0.9837 - val_loss: 0.0363
Epoch 7/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 218s 2s/step - accuracy: 0.9746 - loss: 0.0763 - val_accuracy: 0.9873 - val_loss: 0.0454
Epoch 8/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 180

In [7]:
from tensorflow.keras.preprocessing import image
import numpy as np

# Class names
class_names = list(train_data.class_indices.keys())

def predict_disease(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    
    predictions = model.predict(img_array)
    predicted_class = class_names[np.argmax(predictions)]
    confidence = np.max(predictions) * 100
    
    print(f"🌿 Disease Detected: {predicted_class}")
    print(f"🎯 Confidence: {confidence:.1f}%")
    return predicted_class, confidence

print("✅ Prediction function ready!")
print(f"Can detect: {class_names}")

✅ Prediction function ready!
Can detect: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust']


In [8]:
# Save the model
model.save('crop_disease_model.keras')
print("✅ Model saved as crop_disease_model.keras!")

# Also save class names
import json
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)
print("✅ Class names saved!")
print(f"Classes: {class_names}")

✅ Model saved as crop_disease_model.keras!
✅ Class names saved!
Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust']


In [9]:
# Test with a sample image from dataset
import os

# Get one sample image
sample_dir = train_path + "/Apple___Apple_scab"
sample_img = os.listdir(sample_dir)[0]
sample_path = f"{sample_dir}/{sample_img}"

print(f"Testing with image: {sample_img}")
predict_disease(sample_path)

Testing with image: 00075aa8-d81a-4184-8541-b692b78d398a___FREC_Scab 3335.JPG
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
🌿 Disease Detected: Apple___Cedar_apple_rust
🎯 Confidence: 84.7%


('Apple___Cedar_apple_rust', np.float32(84.71841))

In [10]:
import json

# Save the model
model.save('crop_disease_model.keras')
print("✅ Model saved!")

# Save class names
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)
print("✅ Class names saved!")
print(f"Classes saved: {class_names}")

✅ Model saved!
✅ Class names saved!
Classes saved: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust']
